In [ ]:
from pyprojroot import here
import sys
sys.path.insert(0, str(here()))

In [ ]:
from src.evaluation import run_evaluation
import pandas as pd

In [ ]:
scenarios = [
    {"name": "Altimeter + Compass", "use_altimeter": True,  "use_compass": True},
    {"name": "Altimeter Only",     "use_altimeter": True,  "use_compass": False},
    {"name": "Compass Only",       "use_altimeter": False, "use_compass": True},
    {"name": "No Sensors",         "use_altimeter": False, "use_compass": False},
]

In [ ]:
rows = []
for config in scenarios:
    _, s = run_evaluation(
        ground_truth_path=str(gt_path),
        db_path=str(db_path),
        masks_dir=str(masks_dir),
        use_altimeter=config["use_altimeter"],
        use_compass=config["use_compass"],
        weights=(0.33, 0.33, 0.33),
        min_std_deg=1.5,
        min_max_elev_deg=1.0
    )
    rows.append({
        "Configuration": config["name"],
        "Top-1 Accuracy (500m)": s["top1_acc_500m"],
        "Median Error (m)": s["median_error_m"],
        "Filtered Queries": s["skipped_flat"]
    })


In [ ]:
df_ablation = pd.DataFrame(rows)
df_ablation

In [ ]:
plt.figure(figsize=(8, 4))
bars = plt.bar(df_ablation["Configuration"], df_ablation["Top-1 Accuracy (500m)"], 
               color="steelblue", edgecolor="black", width=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f"{yval:.1f}%", 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.title("Relocalization Accuracy vs. Available Sensors", fontsize=12, fontweight='bold')
plt.ylabel("Top-1 Accuracy @ 500m (%)", fontsize=10)
plt.ylim(0, 110)
plt.grid(True, axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()